以下是基于 NicheCompass 原始代码运行指定乳腺癌数据集的完整代码

In [ ]:
import sys
import os
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. 添加NicheCompass原始代码路径到Python环境（关键）
nichecompass_path = "/home/zhangjunyi/xiangmu/nichecompass-main/src/nichecompass_copy"  # 你的NicheCompass原始代码目录
sys.path.append(nichecompass_path)

# 导入NicheCompass核心模块（需匹配原始代码的实际模块名，以下为通用命名）
from model import NicheCompass  # 核心模型类
from utils import preprocess_data, cal_spatial_neighbors  # 工具函数（根据原始代码调整）

# 2. 配置路径
data_path = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
result_save_path = "./nichecompass_breast_cancer_results"  # 结果保存目录
os.makedirs(result_save_path, exist_ok=True)

# 3. 加载数据集
print("加载数据集...")
adata = ad.read_h5ad(data_path)

# 4. 数据预处理（适配NicheCompass要求，根据原始代码调整）
## 检查并补全必要字段：空间坐标(obsm['spatial'])、细胞类型(obs['cell_type'])
if "spatial" not in adata.obsm.keys():
    raise ValueError("数据集缺少空间坐标（obsm['spatial']），请检查数据格式！")
if "cell_type" not in adata.obs.columns:
    raise ValueError("数据集缺少细胞类型标注（obs['cell_type']），请检查数据格式！")

## 标准化表达矩阵（NicheCompass常规预处理）
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)  # 筛选高变基因

## 计算空间邻域（NicheCompass核心输入）
print("计算空间邻域...")
adata = cal_spatial_neighbors(
    adata,
    spatial_key="spatial",
    n_neighbors=15,  # 邻域数量（可根据数据调整）
    radius=None  # 若为基于半径的邻域，可设置具体值，否则用k近邻
)



In [ ]:
# 5. 初始化NicheCompass模型
print("初始化NicheCompass模型...")
nc_model = NicheCompass(
    adata=adata,
    cell_type_key="cell_type",  # 细胞类型字段名
    spatial_neighbor_key="spatial_neighbors",  # 空间邻域字段名（匹配cal_spatial_neighbors输出）
    n_latent=32,  # 隐层维度（可调整）
    n_epochs=100,  # 训练轮数（可调整）
    batch_size=64,  # 批次大小
    lr=1e-3  # 学习率
)

# 6. 训练模型并识别生态位
print("训练模型并识别生态位...")
adata = nc_model.fit(
    save_loss=True,  # 保存训练损失
    loss_save_path=os.path.join(result_save_path, "training_loss.png")
)



In [ ]:
## 核心步骤：生态位识别（根据原始代码的方法名调整，如identify_niches）
adata = nc_model.identify_niches(
    n_niches=None,  # 自动推断生态位数量；也可手动设置（如5/6）
    clustering_method="leiden"  # 聚类方法（leiden/kmeans，匹配原始代码）
)

# 7. 保存结果
print("保存结果...")
## 保存带生态位标注的adata
adata.write_h5ad(os.path.join(result_save_path, "breast_cancer_niche_annotated.h5ad"))

## 保存生态位统计信息
niche_stats = adata.obs[["cell_type", "niche"]].value_counts().reset_index()
niche_stats.columns = ["cell_type", "niche_id", "count"]
niche_stats.to_csv(os.path.join(result_save_path, "niche_cell_type_stats.csv"), index=False)

# 8. 可视化生态位结果（空间分布+细胞类型组成）
print("绘制生态位可视化结果...")
plt.rcParams["figure.figsize"] = (12, 5)

## 子图1：生态位空间分布
plt.subplot(1, 2, 1)
sc.pl.spatial(
    adata,
    color="niche",
    title="Niche Spatial Distribution",
    frameon=False,
    spot_size=50  # 点大小（根据数据调整）
)

## 子图2：各生态位的细胞类型组成
plt.subplot(1, 2, 2)
niche_cell_type = pd.crosstab(adata.obs["niche"], adata.obs["cell_type"])
niche_cell_type = niche_cell_type.div(niche_cell_type.sum(axis=1), axis=0)  # 归一化
niche_cell_type.plot(kind="bar", stacked=True, ax=plt.gca())
plt.title("Cell Type Composition of Niches")
plt.xlabel("Niche ID")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

## 保存可视化图
plt.savefig(os.path.join(result_save_path, "niche_visualization.png"), dpi=300, bbox_inches="tight")
plt.close()

print("所有步骤完成！结果已保存至：", result_save_path)